# 021 原 ResNet 对照
这一组恢复原代码中的 ResNet50 双分支结构和优化器参数，用来与02当前模型比较。网络定义和需要的函数都直接写在这本 Notebook 中。

使用与02相同的四层缓存、患者清单和五折划分。结果保存到 `results/resnet_original`；02的结果继续放在 `results/resnet`。


## 这一组具体恢复什么
| 项目 | 021原网络 | 02当前模型 |
| --- | --- | --- |
| 每层影像特征 | ResNet50输出2048维 | 压缩为256维 |
| 病灶分支 | BatchNorm，输出256维 | GroupNorm，输出64维 |
| 四层注意力 | Linear → Tanh → Dropout → Linear | 门控注意力 |
| 主干训练范围 | 从第1轮开始训练整个ResNet，BatchNorm正常更新 | 预热3轮后仅微调最后一组残差块，BatchNorm固定 |
| 主干学习率 | 1e-4 | 1e-5 |
| 新增部分学习率 | 5e-4 | 5e-4 |

批量16、最多30轮、连续8轮内部验证AUC不提升就停止，以及原来的数据增强、加权BCE、AdamW、权重衰减1e-4和梯度裁剪均保留。

原网络结构和增强取自 `code/train_resnet50_manual_cv.py`，学习率也与原时间划分训练脚本一致。训练时加载ImageNet权重，读取训练结果时直接加载本实验权重。


## 与02如何比较
评估方式使用02目前的流程：每折先在外层训练患者内部划出20%选择轮数，再用该折全部训练患者重训，最后预测留出的患者。随机种子、分组、阈值选择和概率计算也与02一致。

因此这组比较的是“原结构＋原训练参数”和“02调整方案”。它并非逐步复现以前的0.81/0.75：旧脚本在外层验证患者上选择轮数，新流程把选轮数放进训练集内部。时间测试队列留给06统一测试。


## 数据位置与运行参数


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config.json").exists() and (ROOT / "1117" / "config.json").exists():
    ROOT = ROOT / "1117"
cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))

def path_from_config(key):
    value = Path(cfg[key]).expanduser()
    return value if value.is_absolute() else ROOT / value

CACHE = path_from_config("cache_path")
COHORT = path_from_config("cohort_path")
RESULTS = path_from_config("results_dir")
SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]
print("患者清单：", COHORT)
print("结果目录：", RESULTS)


In [ ]:
def read_cases():
    cases = pd.read_csv(COHORT, dtype={"mr": str})
    if not cases.mr.is_unique or not set(cases.split) <= {"development", "validation"}:
        raise ValueError("患者编号重复或分组有误，请检查cohort.csv。")
    if not set(cases.label) <= {0, 1} or cases.mr.isna().any():
        raise ValueError("患者编号缺失或结局不是0/1。")
    return cases


def read_classification_data():
    cases = read_cases()
    with np.load(CACHE, allow_pickle=False) as data:
        cache_mrs = data["mrs"].astype(str).tolist()
        if len(set(cache_mrs)) != len(cache_mrs) or set(cache_mrs) != set(cases.mr):
            raise ValueError("缓存与患者清单的MR编号不一致。")
        lookup = {mr: i for i, mr in enumerate(cache_mrs)}
        order = [lookup[mr] for mr in cases.mr]
        if not np.array_equal(data["labels"][order], cases.label) or not np.array_equal(data["max_layers"][order], cases.max_layer):
            raise ValueError("患者标签或取层与缓存不一致。")
        images = data["images"][order].astype(np.float32)
        masks = data["masks"][order].astype(np.uint8)
    if images.shape[1:] != (4, 3, 224, 224) or masks.shape != images.shape:
        raise ValueError("分类输入应为每例4层、3通道、224×224。")
    if not np.isfinite(images).all() or not np.isin(masks, [0, 1]).all():
        raise ValueError("影像存在无效值或mask不是0/1。")
    return cases, images, masks


In [ ]:
cases, images, masks = read_classification_data()
OUTPUT = RESULTS / "resnet_original"
TRAIN = {
    "batch_size": 16,
    "max_epochs": 30,
    "patience": 8,
    "backbone_lr": 1e-4,
    "head_lr": 5e-4,
}
RUN_TRAINING = False
RUN_CLINICAL = False
RESTART = False

display(cases.groupby("split").agg(n=("mr", "size"), positive=("label", "sum")))
print("原网络结果目录：", OUTPUT)


## 原网络结构
ResNet50处理每层三序列影像，病灶编码器处理对应的三个标签通道。2048维影像特征与256维病灶特征拼接为2304维，再通过普通注意力汇总四层，输出患者级概率。

这里没有影像特征压缩、门控注意力、主干冻结或预热阶段。即使选出的训练轮数为1，ResNet主干也已经参与这一轮训练。


In [ ]:
import random
import torch
from torch import nn
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("计算设备：", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class OriginalMaskEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x.float())


class OriginalResNet(nn.Module):
    def __init__(self, pretrained=False) -> None:
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.mask_features = OriginalMaskEncoder()
        self.attention = nn.Sequential(
            nn.Linear(2304, 256),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(2304),
            nn.Dropout(0.4),
            nn.Linear(2304, 1),
        )

    def forward(self, images: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        B, S, C, H, W = images.shape
        im = images.view(B * S, C, H, W)
        image_feat = self.image_features(im)
        image_feat = torch.flatten(image_feat, 1)
        image_feat = image_feat.view(B, S, -1)

        mask = masks.view(B * S, 3, H, W)
        mask_feat = self.mask_features(mask)
        mask_feat = mask_feat.view(B, S, -1)

        feat = torch.cat([image_feat, mask_feat], dim=-1)
        attn = self.attention(feat).squeeze(-1)
        attn = torch.softmax(attn, dim=1)
        pooled = (feat * attn.unsqueeze(-1)).sum(dim=1)
        return self.head(pooled).squeeze(-1)


def sequence_input(images, sequence="all"):
    if sequence == "all":
        return images
    channel = SEQUENCES.index(sequence)
    return images[:, :, [channel, channel, channel]]


@torch.no_grad()
def predict(model, images, masks, batch_size=16):
    model.eval()
    probabilities = []
    for start in range(0, len(images), batch_size):
        x = torch.as_tensor(images[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        m = torch.as_tensor(masks[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(x, m)
        probabilities.append(logits.float().sigmoid().cpu().numpy())
    return np.concatenate(probabilities)


## 增强与指标
四层影像和标签同步做几何变换；仅影像增加亮度扰动和噪声。开发队列的折外概率用于计算AUC并确定分类阈值。


In [ ]:
def augment_batch(images: torch.Tensor, masks: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    out_images = images.clone()
    out_masks = masks.clone()
    B, S = images.shape[:2]
    for b in range(B):
        angle = random.uniform(-6.0, 6.0)
        translate = (random.uniform(-6.0, 6.0), random.uniform(-6.0, 6.0))
        scale = random.uniform(0.96, 1.04)
        for s in range(S):
            out_images[b, s] = affine(
                images[b, s],
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            )
            out_masks[b, s] = affine(
                masks[b, s].float(),
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.NEAREST,
                fill=0.0,
            ).byte()

    jitter = 0.9 + 0.2 * torch.rand(B, 1, 1, 1, 1, device=out_images.device)
    out_images = out_images * jitter
    if random.random() < 0.5:
        out_images = out_images + torch.randn_like(out_images) * 0.008
    out_images = out_images.clamp(0.0, 1.0)
    return out_images, out_masks


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss


def choose_threshold(labels, probabilities):
    fpr, tpr, thresholds = roc_curve(labels, probabilities)
    finite = np.isfinite(thresholds)
    return float(thresholds[finite][np.argmax((tpr - fpr)[finite])])


def classification_metrics(labels, probabilities, threshold):
    labels, probabilities = np.asarray(labels, int), np.asarray(probabilities, float)
    if not np.isfinite(threshold):
        raise ValueError("分类阈值必须是开发队列确定的有限数值。")
    if set(labels) != {0, 1} or not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("指标需要两类结局，以及0～1之间的有效概率。")
    tn, fp, fn, tp = confusion_matrix(labels, probabilities >= threshold, labels=[0, 1]).ravel()
    return {"n": len(labels), "auc": float(roc_auc_score(labels, probabilities)), "threshold": float(threshold),
            "accuracy": float((tp + tn) / len(labels)), "sensitivity": float(tp / (tp + fn)),
            "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(labels, probabilities))}


## 选择轮数、五折训练和最终模型
日志会分别标明“内部选择轮数”和“重训后评估留出患者”。最终模型使用五折最佳轮数的中位数，在全部开发患者上训练。

每折的 `selection_history.csv` 记录训练损失和内部验证AUC，`training_history.csv` 记录正式重训损失。已完成的折默认复用；当前折中断后重新运行会重做这一折。修改参数后重训同名实验时设 `RESTART=True`，另存实验时修改 `OUTPUT`。


In [ ]:
from sklearn.model_selection import train_test_split


def fit_network(images, masks, labels, train_index, settings, epochs, seed, valid_index=None, history_file=None):
    set_seed(seed)
    model = OriginalResNet(pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": model.image_features.parameters(), "lr": settings["backbone_lr"]},
        {"params": [p for name, p in model.named_parameters() if not name.startswith("image_features.")], "lr": settings["head_lr"]},
    ], weight_decay=1e-4)
    y = labels[train_index]
    pos_weight = torch.tensor([max(float((y == 0).sum()) / max(int((y == 1).sum()), 1), .5)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best_auc, best_epoch, stale = -1.0, 1, 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        shuffled = np.asarray(train_index)[torch.randperm(len(train_index)).numpy()]
        total_loss = 0.0
        for start in range(0, len(shuffled), settings["batch_size"]):
            index = shuffled[start:start + settings["batch_size"]]
            x, m = augment_batch(torch.from_numpy(images[index]), torch.from_numpy(masks[index]))
            yb = torch.as_tensor(labels[index], dtype=torch.float32, device=DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                loss = criterion(model(x.to(DEVICE), m.to(DEVICE)), yb)
            if not torch.isfinite(loss):
                raise ValueError("损失出现无效数值，请检查输入或降低学习率。")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * len(index)
        message = f"epoch {epoch:02d}/{epochs}, loss={total_loss / len(train_index):.4f}"
        row = {"epoch": epoch, "loss": total_loss / len(train_index)}
        if valid_index is not None:
            probability = predict(model, images[valid_index], masks[valid_index], settings["batch_size"])
            auc = float(roc_auc_score(labels[valid_index], probability))
            row["inner_auc"] = auc
            if auc > best_auc:
                best_auc, best_epoch, stale = auc, epoch, 0
            else:
                stale += 1
            message += f", inner_auc={auc:.4f}, best_epoch={best_epoch}"
        history.append(row)
        if history_file is not None:
            pd.DataFrame(history).to_csv(history_file, index=False)
        print(message, flush=True)
        if valid_index is not None and stale >= settings["patience"]:
            break
    return model, best_epoch


def run_experiment(cases, images, masks, folder, settings, sequence="all", restart=False):
    folder.mkdir(parents=True, exist_ok=True)
    dev_index = np.flatnonzero(cases.split.eq("development"))
    dev = cases.iloc[dev_index].reset_index(drop=True)
    x, m = sequence_input(images[dev_index], sequence), masks[dev_index]
    y, folds = dev.label.to_numpy(dtype=int), dev.development_fold.to_numpy(dtype=int)
    if set(folds) != {1, 2, 3, 4, 5}:
        raise ValueError("开发队列需有1～5的固定折号。")
    oof, chosen_epochs, fold_rows, trained = np.zeros(len(dev), np.float32), [], [], False
    for fold in range(1, 6):
        train_index, valid_index = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
        weight_file, prediction_file = folder / f"fold_{fold}.pt", folder / f"fold_{fold}_oof.csv"
        if not restart and weight_file.exists() and prediction_file.exists():
            saved = torch.load(weight_file, map_location="cpu", weights_only=True)
            previous = pd.read_csv(prediction_file, dtype={"mr": str}).set_index("mr")
            if not previous.index.is_unique or set(previous.index) != set(dev.mr.iloc[valid_index]):
                raise ValueError("已有折结果的患者不同；重训请设置RESTART=True。")
            previous = previous.loc[dev.mr.iloc[valid_index]]
            if not np.array_equal(previous.label, y[valid_index]):
                raise ValueError("已有折结果的标签不同；重训请设置RESTART=True。")
            oof[valid_index], epochs = previous.probability.to_numpy(), int(saved["epochs"])
            print(f"fold {fold}：读取已完成结果", flush=True)
        else:
            inner_train, inner_valid = train_test_split(train_index, test_size=.2, stratify=y[train_index], random_state=SEED + fold)
            print(f"fold {fold}：内部选择轮数", flush=True)
            probe, epochs = fit_network(x, m, y, inner_train, settings, settings["max_epochs"], SEED + fold, inner_valid,
                                        history_file=folder / f"fold_{fold}_selection_history.csv")
            del probe
            print(f"fold {fold}：选定{epochs}轮，重训后评估留出患者", flush=True)
            model, _ = fit_network(x, m, y, train_index, settings, epochs, SEED + 100 + fold,
                                   history_file=folder / f"fold_{fold}_training_history.csv")
            oof[valid_index] = predict(model, x[valid_index], m[valid_index], settings["batch_size"])
            torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "architecture": "original_resnet50",
                        "sequence": sequence, "epochs": epochs, "settings": settings,
                        "train_mrs": dev.mr.iloc[train_index].tolist()}, weight_file)
            frame = dev.iloc[valid_index][["mr", "label", "development_fold"]].copy()
            frame["probability"] = oof[valid_index]
            frame.to_csv(prediction_file, index=False)
            del model
            trained = True
        chosen_epochs.append(epochs)
        fold_rows.append({"fold": fold, "auc": float(roc_auc_score(y[valid_index], oof[valid_index])), "epochs": epochs})
    threshold = choose_threshold(y, oof)
    final_file = folder / "model.pt"
    if restart or trained or not final_file.exists():
        final_epochs = max(1, int(np.median(chosen_epochs)))
        print(f"最终模型：全部开发患者训练{final_epochs}轮", flush=True)
        final, _ = fit_network(x, m, y, np.arange(len(dev)), settings, final_epochs, SEED + 1000, history_file=folder / "final_training_history.csv")
        torch.save({"state_dict": {k: v.cpu() for k, v in final.state_dict().items()}, "architecture": "original_resnet50",
                    "sequence": sequence, "threshold": threshold, "epochs": final_epochs, "settings": settings,
                    "train_mrs": dev.mr.tolist()}, final_file)
        del final
    frame = dev[["mr", "label", "development_fold"]].copy()
    frame["probability"] = oof
    frame.to_csv(folder / "development_oof.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(folder / "fold_metrics.csv", index=False)
    metrics = {**classification_metrics(y, oof, threshold), "mean_fold_auc": float(np.mean([r["auc"] for r in fold_rows]))}
    pd.DataFrame([metrics]).to_csv(folder / "development_metrics.csv", index=False)
    return pd.DataFrame([metrics])


In [ ]:
if RUN_TRAINING:
    display(run_experiment(cases, images, masks, OUTPUT, TRAIN, restart=RESTART))
else:
    print("将RUN_TRAINING改为True，运行本节开始原网络对照。")


## 临床对照与原网络融合
使用与02相同的五项临床变量和逻辑回归设置。融合输入来自021自己的影像折外概率，结果名为 `OriginalResNet_Clinical5`。

临床单独模型仍叫 `Clinical5`。在相同患者和参数下，它应与02的临床对照一致。融合仍是一轮折外预测再做逻辑回归，开发结果不是完整嵌套的融合评估。影像模型重训后，需要重跑本节融合。


In [ ]:
import joblib
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold


def clinical_search():
    pipeline = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                         ("scale", StandardScaler()), ("lr", LogisticRegression(solver="liblinear", max_iter=3000))])
    return GridSearchCV(pipeline, {"lr__C": [.003, .01, .03, .1, .3, 1.]}, scoring="roc_auc",
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=1, error_score="raise")


def fit_clinical_models(cases, resnet_folder):
    dev = cases.query("split == 'development'").reset_index(drop=True)
    image_oof = pd.read_csv(resnet_folder / "development_oof.csv", dtype={"mr": str}).set_index("mr")
    if not image_oof.index.is_unique or set(image_oof.index) != set(dev.mr):
        raise ValueError("影像OOF与开发患者不一致。")
    if not np.array_equal(image_oof.loc[dev.mr, "label"], dev.label):
        raise ValueError("影像OOF的结局与患者清单不一致。")
    x_clinical = dev[CLINICAL].to_numpy(dtype=float)
    x_fused = np.column_stack([image_oof.loc[dev.mr, "probability"].to_numpy(), x_clinical])
    y, folds = dev.label.to_numpy(dtype=int), dev.development_fold.to_numpy(dtype=int)
    models, rows, tables = {}, [], []
    for name, values in (("Clinical5", x_clinical), ("OriginalResNet_Clinical5", x_fused)):
        probabilities = np.zeros(len(dev), dtype=float)
        for fold in range(1, 6):
            valid = folds == fold
            search = clinical_search().fit(values[~valid], y[~valid])
            probabilities[valid] = search.predict_proba(values[valid])[:, 1]
        threshold = choose_threshold(y, probabilities)
        models[name] = {"model": clinical_search().fit(values, y).best_estimator_, "threshold": threshold}
        rows.append({"model": name, **classification_metrics(y, probabilities, threshold)})
        table = dev[["mr", "label", "development_fold"]].copy()
        table["model"], table["probability"] = name, probabilities
        tables.append(table)
    joblib.dump(models, resnet_folder / "clinical_models.joblib")
    pd.DataFrame(rows).to_csv(resnet_folder / "clinical_development_metrics.csv", index=False)
    pd.concat(tables).to_csv(resnet_folder / "clinical_development_oof.csv", index=False)
    return pd.DataFrame(rows)


In [ ]:
if RUN_CLINICAL:
    display(fit_clinical_models(cases, OUTPUT))
else:
    print("影像五折训练完成后，将RUN_CLINICAL改为True，运行临床对照与融合。")


## 查看02与021的开发结果
读取已经生成的指标并排显示，不重跑训练。先比较整体折外AUC和各折表现，再看融合是否带来增益；最终效果在06中使用同一时间测试队列比较。


In [ ]:
image_results, clinical_results = [], []
for experiment, folder in (("02_adjusted", RESULTS / "resnet"), ("021_original", OUTPUT)):
    file = folder / "development_metrics.csv"
    if file.exists():
        table = pd.read_csv(file)
        table.insert(0, "experiment", experiment)
        image_results.append(table)
    file = folder / "clinical_development_metrics.csv"
    if file.exists():
        table = pd.read_csv(file)
        table.insert(0, "experiment", experiment)
        clinical_results.append(table)
if image_results:
    display(pd.concat(image_results, ignore_index=True).round(4))
if clinical_results:
    display(pd.concat(clinical_results, ignore_index=True).round(4))
if (OUTPUT / "fold_metrics.csv").exists():
    display(pd.read_csv(OUTPUT / "fold_metrics.csv"))


021主要输出与02相同：`model.pt`、`development_metrics.csv`、`fold_metrics.csv`、`development_oof.csv` 和 `clinical_models.joblib`。更新后的06会从 `results/resnet_original` 读取权重及融合模型，无需导入清单或额外JSON。
